In [ ]:
import numpy as np
import pylab as plt
import torch
import torch.nn as nn
from mmengine import Config
from AI4AO.DAO_Utils import *
from AI4AO import PSFDataset, imshow, imshow_multiple
from AI4AO.paths import instrument_dir
from AI4AO.TorchPropagator import WFS
from tqdm import tqdm
import torch.nn.functional as F

device = 'cuda'

In [ ]:
WFSParams = dict(
    {
        "Nres":36,
        "sampling":120/36,
        "D": 1.5,
        "centralObstruction": 0.,
        "useNoise": True,
        "Modulation": 0,
        "Wavelength": 635e-9,
        "Nphotons": [5, 7],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.04, 0.12],  # Fired parameter range (m)
        "L0": [10, 30.0000],  # Outter scale range (m)
        "Nphases": 16,  # Number of phases in the batch
        "Layers": [5, 10],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 2,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [0.97, 1.],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.] 
    }
)

## Atmosphere parameters
DMParams = dict(
    {
        "Nactuator": 11,
    }
)


## PSF parameters (PSFDataset)
# sampling and fov are [min, max] ranges: each batch draws one value of each, shared by its
# PSFs, so the PSF (and satellite image) size round(fov * sampling) changes between batches.
PSFParams = dict(
    {
        "Wavelength": 635e-9,     # science wavelength (m)
        "sampling": 3,       # PSF pixels per lambda/D (>= 2; steps of 2 / Nres)
        "fov": [30, 36],          # field of view (lambda/D, at most Nres = 36)
        "NCPA": [10e-9, 40e-9],   # static aberration rms range (m)
        "Jitter": [1.0, 5.0],     # residual jitter rms per axis (mas)
        "NoisePhotonFactor": 1 / 4,
        "NoiseRONFactor": 1.0,
    }
)
# Error terms: fitting + servo reproduce the closed-loop PhaseDataset PSFs used so far.
# For more realistic PSFs add "aliasing", "noise", "static", "jitter" -- "noise" then needs
# PSFParams["NoisePhotonFactor"] / ["NoiseRONFactor"], and "static"/"jitter" use
# PSFParams["NCPA"] (m rms range) / ["Jitter"] (mas rms range). See Tutorials/basics/06_PSFDataset.ipynb.
psf_terms = ("fitting", "servo", "aliasing", "noise", "static", "jitter")

In [ ]:
# WFS creation (PSFDataset uses its pupil and GetPSF convention)
wfs = WFS(WFSParams, device)

In [ ]:
# Long-exposure PSFs straight from the residual PSD: one converged PSF per sample,
# instead of summing 20-100 GetPSF frames of PhaseDataset screens
psf_dataset = PSFDataset(wfs, WFSParams, AtmosParams, LoopParams, DMParams, PSFParams, device, terms=psf_terms)

In [ ]:
batch = psf_dataset[0]
psf_frame = batch["psf"]
imshow(psf_frame.sqrt())
plt.show()

In [ ]:
def GenerateSateliteImage(input: torch.tensor, number_of_squares:int = 5):
    C,W,H = input.shape
    x = torch.linspace(-W//2, W//2, W, device=input.device, dtype=input.dtype)
    y = torch.linspace(-H//2, H//2, H, device=input.device, dtype=input.dtype)
    x,y = torch.meshgrid(x,y)
    angles = torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype)*2*torch.pi
    translation_x = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 4
    translation_y = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 4
    scale_x = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 2
    scale_y = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 2
    brightness = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype)) * scale_x * scale_y
    translation_phases = torch.exp(1j*2*torch.pi*(x * translation_x + y * translation_y))
    sinc_x = torch.sinc(torch.cos(angles)*x*scale_x + torch.sin(angles)*y*scale_y)
    sinc_y = torch.sinc(-torch.sin(angles)*x*scale_x + torch.cos(angles)*y*scale_y)
    PSD = sinc_x * sinc_y * translation_phases * brightness
    return torch.abs(torch.fft.fftshift(torch.fft.fft2(PSD))).sum(dim=0)

In [ ]:
test = torch.zeros((psf_frame.shape[0], 100, 100), device=device, dtype=torch.float32)

test = GenerateSateliteImage(test, 10)
imshow(test)
plt.show()

In [ ]:
conv_test = torch.conv2d(test.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
imshow(conv_test)
plt.show()

In [ ]:
batch = psf_dataset[0]
psf_frame = batch["psf"]
wfs.SetPhotonsAndRON(batch["nphotons"].view(-1, 1, 1), batch["ron"].view(-1, 1, 1))

test = torch.zeros_like(psf_frame)
satelite_GT = GenerateSateliteImage(test, 10)

satelite_blured = torch.conv2d(satelite_GT.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
satelite_blured /= satelite_blured.sum(dim = (-2,-1), keepdim=True)
satelite_blured = wfs.AddNoiseToFrame(satelite_blured)

imshow_multiple([
    {"tensor": psf_frame.sqrt()},
    {"tensor": satelite_GT},
    {"tensor": satelite_blured},
    ],
    )
plt.show()

In [ ]:
def make_satelite_frame(psf_dataset):
    batch = psf_dataset[0]
    psf_frame = batch["psf"]
    wfs.SetPhotonsAndRON(batch["nphotons"].view(-1, 1, 1), batch["ron"].view(-1, 1, 1))

    psf_frame /= psf_frame.mean(dim = (-2,-1), keepdim=True)
    test = torch.zeros_like(psf_frame)
    satelite_GT = GenerateSateliteImage(test, 10)
    satelite_GT /= satelite_GT.mean(dim = (-2,-1), keepdim=True)

    satelite_blured = torch.conv2d(satelite_GT.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
    satelite_blured /= satelite_blured.sum(dim = (-2,-1), keepdim=True)
    satelite_blured_noisy = wfs.AddNoiseToFrame(satelite_blured)
    satelite_blured /= satelite_blured.mean(dim = (-2,-1), keepdim=True)
    satelite_blured_noisy /= satelite_blured_noisy.mean(dim = (-2,-1), keepdim=True)

    psf_frame = psf_frame.sqrt()
    psf_frame /= psf_frame.mean(dim = (-2,-1), keepdim=True)

    return psf_frame, satelite_GT, satelite_blured.permute(1,0,2,3), satelite_blured_noisy.permute(1,0,2,3)

In [ ]:
def _group_norm_groups(channels, max_groups=8):
    """Largest group count in {8,4,2,1} that evenly divides channels, so
    ConvBlock stays valid for any base_channels the growth knobs are set to."""
    for g in (max_groups, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1

def pad_to_multiple(x, depth):
    multiple = 2 ** depth
    H, W = x.shape[-2:]
    pad_h = (multiple - H % multiple) % multiple
    pad_w = (multiple - W % multiple) % multiple
    return F.pad(x, (0, pad_w, 0, pad_h))

def depad_to_multiple(x, W, H):
    return x[..., :H, :W]



class ConvBlock(nn.Module):
    """Two 3x3 convs (GroupNorm + SiLU each) at a fixed channel count."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.ELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.ELU(),
        )

    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_channels, out_channels,kernel_size=2,stride=2)
        self.conv = ConvBlock(out_channels * 2,out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = ConvBlock(in_channels,out_channels)
        self.down = nn.Conv2d(out_channels, out_channels,kernel_size=2,stride=2)

    def forward(self, x):
        x = self.conv(x)
        return self.down(x), x


class DecoderHead(nn.Module):
    def __init__(self, bottleneck_channels, depth, enc_channels):
        super().__init__()

        self.up_blocks = nn.ModuleList([
            DecoderBlock(bottleneck_channels if i == depth - 1 else enc_channels[i + 1], enc_channels[i])
            for i in reversed(range(depth))
        ])

        self.head = nn.Conv2d(enc_channels[0], 1, kernel_size=1)

    def forward(self, x, skips):
        for block, skip in zip(self.up_blocks, reversed(skips)):
            x = block(x, skip)
        
        raw = F.softplus(self.head(x))              # (Nphases,1,H,W), non-negative
        frame = raw / raw.mean(dim=(-2, -1), keepdim=True)
        return frame

class EncoderHead(nn.Module):
    def __init__(self, bottleneck_channels, depth, enc_channels):
        super().__init__()

        self.down_blocks = nn.ModuleList([
            EncoderBlock(1 if i == 0 else enc_channels[i - 1], enc_channels[i])
            for i in range(depth)
        ])

    def forward(self, x):
            """frame_stack: (Nphases, n_frames, H, W), oldest first."""
            skips = []
            for block in self.down_blocks:
                x, skip = block(x)
                skips.append(skip)
            return x, skips

class DeconvolutionUNet(nn.Module):
    def __init__(self, depth=3, base_channels=16):
        super().__init__()
        self.depth = depth
        enc_channels = [base_channels * 2 ** i for i in range(depth)]
        bottleneck_channels = enc_channels[-1] * 2

        self.encoder = EncoderHead(bottleneck_channels, depth, enc_channels)
        self.bottleneck = ConvBlock(enc_channels[-1], bottleneck_channels)
        self.decoder_frame = DecoderHead(bottleneck_channels, depth, enc_channels)
        self.decoder_psf = DecoderHead(bottleneck_channels, depth, enc_channels)

    def forward(self, x):
        # Original size and pad
        H, W = x.shape[-2:]
        x = pad_to_multiple(x, self.depth)

        # Network
        x, skips = self.encoder(x)
        x = self.bottleneck(x)

        frame = self.decoder_frame(x, skips)
        psf = self.decoder_psf(x, skips)

        # Remove padding
        frame = depad_to_multiple(frame, W, H)
        psf = depad_to_multiple(psf, W, H)

        return frame.squeeze(1), psf.squeeze(1)                     # (Nphases,H,W), matches wfs frame convention

In [ ]:
unet = DeconvolutionUNet(depth=4, base_channels=32).to(device)


n_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")

try:
    unet.load_state_dict(torch.load(instrument_dir("Deconvolution") / "11MUNet.pth"))
except:
    print("Starting from scratch")

In [ ]:
loss_trackers = {}
steps = 3000

optimizer = torch.optim.AdamW(unet.parameters(), 1e-4, fused=(device == 'cuda'))
loss_tracker = torch.zeros(steps, device=device)

unet.train()
progressBar = tqdm(range(steps))

for step in progressBar:
    with torch.no_grad():
        psf, satelite_GT, satelite_blured, satelite_blured_noisy = make_satelite_frame(psf_dataset)

    # deconvolved_frame = unet(satelite_blured)
    deconvolved_frame, extracted_psf = unet(satelite_blured_noisy)

    reconvolved_frame = torch.conv2d(deconvolved_frame.unsqueeze(0), extracted_psf.square().unsqueeze(1), groups=psf_frame.shape[0], padding="same")
    reconvolved_frame /= reconvolved_frame.mean(dim = (-2,-1), keepdim=True)
    reconvolved_frame = reconvolved_frame.squeeze()

    loss = F.l1_loss(deconvolved_frame, satelite_GT) + F.l1_loss(extracted_psf, psf) / 2 + F.l1_loss(reconvolved_frame, satelite_blured.squeeze()) / 10

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    loss_tracker[step] = loss.detach()

    if step % 100 == 1:
        lower = max(0, step - 100)
        progressBar.set_postfix({'Loss': float(loss_tracker[lower:step].mean())})

    


In [ ]:
torch.save(unet.state_dict(), instrument_dir("Deconvolution") / "11MUNet.pth")

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

plt.loglog(smooth(loss_tracker.cpu()))

In [ ]:
F.mse_loss(deconvolved_frame, satelite_GT), F.mse_loss(extracted_psf, psf), F.mse_loss(reconvolved_frame, satelite_blured.squeeze())

In [ ]:
psf, satelite_GT, satelite_blured, satelite_blured_noisy = make_satelite_frame(psf_dataset)
deconvolved_frame, extracted_psf = unet(satelite_blured_noisy)

imshow(torch.stack(
    (satelite_blured_noisy.squeeze(), satelite_GT.squeeze(), deconvolved_frame.squeeze(), satelite_GT.squeeze() - deconvolved_frame.squeeze()),
    dim = 1)
    )
plt.show()
plt.figure()
imshow(torch.stack(
        (psf.squeeze(), extracted_psf.squeeze(), psf.squeeze() - extracted_psf.squeeze()),
        dim = 1
        ))

plt.show()

In [ ]:
reconvolved_frame = torch.conv2d(deconvolved_frame.unsqueeze(0), extracted_psf.square().unsqueeze(1), groups=psf_frame.shape[0], padding="same")
reconvolved_frame /= reconvolved_frame.mean(dim = (-2,-1), keepdim=True)
reconvolved_frame = reconvolved_frame.squeeze()

imshow(torch.stack((satelite_blured.squeeze(), reconvolved_frame, satelite_blured.squeeze() - reconvolved_frame), dim = 1))
plt.show()